In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os



In [2]:
load_dotenv("/home/jovyan/work/genAI/.env")

True

In [3]:
client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print(client_openai)
print("="*15)

# --- API CALLING ---

In [4]:
user_input = input("user: ")
response_openai = client_openai.responses.create(model='gpt-5.6-sol', input=user_input)
print(f"""openai: {response_openai.output_text}""")

user:  capital of france is ?


openai: The capital of France is **Paris**.


# --- CONNECTING DB ---

regular steps to interact with db
1. input user question
2. create query of response
3. interact with db
4. run ruery
5. share db response to user

In [15]:
import psycopg
import os

conn = psycopg.connect(
    host="postgres",
    port=5432,
    dbname="genAI",
    user="admin",
    password=os.getenv("POSTGRES_PASSWORD"),
    connect_timeout=10 # Throws an error after 10 seconds if unreachable
)
print("Connected!")



conn.autocommit = True

Connected!


# --- QUERY DB ---

In [16]:
# Create a cursor and execute the query to fetch table names
with conn.cursor() as cur:
    cur.execute("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public'
    """)
    
    # Fetch all the results
    tables = cur.fetchall()
    
    # Print the tables
    if tables:
        print("Tables in the database:")
        for table in tables:
            print(f"- {table[0]}")
    else:
        print("No tables found in the public schema.")

No tables found in the public schema.


# --- CREATE TABLES ---

In [18]:
with conn.cursor() as cur:
    cur.execute("""
        -- 1. Departments Table
        CREATE TABLE IF NOT EXISTS departments (
            department_id SERIAL PRIMARY KEY,
            department_name VARCHAR(100) NOT NULL,
            location VARCHAR(100) NOT NULL
        );

        -- 2. Employees Table
        CREATE TABLE IF NOT EXISTS employees (
            employee_id SERIAL PRIMARY KEY,
            first_name VARCHAR(50) NOT NULL,
            last_name VARCHAR(50) NOT NULL,
            email VARCHAR(100) UNIQUE NOT NULL,
            department_id INT REFERENCES departments(department_id),
            salary NUMERIC(10, 2) NOT NULL,
            hire_date DATE NOT NULL
        );

        -- 3. Customers Table
        CREATE TABLE IF NOT EXISTS customers (
            customer_id SERIAL PRIMARY KEY,
            company_name VARCHAR(150) NOT NULL,
            contact_name VARCHAR(100) NOT NULL,
            country VARCHAR(50) NOT NULL,
            signup_date DATE NOT NULL
        );

        -- 4. Sales Table
        CREATE TABLE IF NOT EXISTS sales (
            sale_id SERIAL PRIMARY KEY,
            customer_id INT REFERENCES customers(customer_id),
            employee_id INT REFERENCES employees(employee_id),
            product_name VARCHAR(100) NOT NULL,
            quantity INT NOT NULL,
            unit_price NUMERIC(10, 2) NOT NULL,
            sale_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );

        -- 5. Shipping Table
        CREATE TABLE IF NOT EXISTS shipping (
            shipment_id SERIAL PRIMARY KEY,
            sale_id INT REFERENCES sales(sale_id),
            carrier VARCHAR(50) NOT NULL,
            shipping_cost NUMERIC(10, 2) NOT NULL,
            delivery_status VARCHAR(50) NOT NULL,
            shipped_date DATE,
            delivered_date DATE
        );

        -- Insert Sample Data: Departments
        INSERT INTO departments (department_name, location) VALUES
        ('Sales & Growth', 'New York'),
        ('Logistics & Ops', 'Chicago'),
        ('Engineering', 'San Francisco')
        ON CONFLICT DO NOTHING;

        -- Insert Sample Data: Employees
        INSERT INTO employees (first_name, last_name, email, department_id, salary, hire_date) VALUES
        ('Alice', 'Smith', 'alice.smith@company.com', 1, 85000.00, '2022-03-15'),
        ('Bob', 'Jones', 'bob.jones@company.com', 2, 72000.00, '2021-06-01'),
        ('Charlie', 'Brown', 'charlie.brown@company.com', 3, 115000.00, '2020-01-10')
        ON CONFLICT DO NOTHING;

        -- Insert Sample Data: Customers
        INSERT INTO customers (company_name, contact_name, country, signup_date) VALUES
        ('Alpha Corp', 'Jane Doe', 'USA', '2023-01-10'),
        ('Beta LLC', 'John Roe', 'Canada', '2023-05-20'),
        ('Gamma Inc', 'Acme User', 'UK', '2024-02-14')
        ON CONFLICT DO NOTHING;

        -- Insert Sample Data: Sales
        INSERT INTO sales (customer_id, employee_id, product_name, quantity, unit_price, sale_date) VALUES
        (1, 1, 'Enterprise SaaS License', 5, 1200.00, '2025-01-10 10:30:00'),
        (2, 1, 'Cloud Hosting Add-on', 10, 150.00, '2025-01-12 14:00:00'),
        (3, 2, 'API Analytics Module', 2, 4500.00, '2025-01-15 09:15:00'),
        (1, 3, 'Security Audit Tool', 1, 3000.00, '2025-01-18 16:45:00')
        ON CONFLICT DO NOTHING;

        -- Insert Sample Data: Shipping
        INSERT INTO shipping (sale_id, carrier, shipping_cost, delivery_status, shipped_date, delivered_date) VALUES
        (1, 'FedEx', 45.00, 'Delivered', '2025-01-11', '2025-01-13'),
        (2, 'UPS', 25.50, 'Delivered', '2025-01-13', '2025-01-16'),
        (3, 'DHL', 120.00, 'In Transit', '2025-01-16', NULL),
        (4, 'FedEx', 50.00, 'Pending', NULL, NULL)
        ON CONFLICT DO NOTHING;
    """)
    print("Complex analytical tables and sample data created successfully!")

Complex analytical tables and sample data created successfully!


In [20]:
# Create a cursor and execute the query to fetch table names
with conn.cursor() as cur:
    cur.execute("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public'
    """)
    
    # Fetch all the results
    tables = cur.fetchall()
    
    # Print the tables
    if tables:
        print("Tables in the database:")
        for table in tables:
            print(f"- {table[0]}")
    else:
        print("No tables found in the public schema.")

Tables in the database:
- departments
- employees
- customers
- sales
- shipping


# --- USE AI TO CREATE A SQL QUERY ---

In [40]:
import pandas as pd

def show_table(cur):
    """Fetches results from the active cursor, converts to a styled DataFrame, and displays it with borders."""
    output = cur.fetchall()
    columns = [desc[0] for desc in cur.description]
    df = pd.DataFrame(output, columns=columns)
    
    # Apply grid borders styling
    styled_df = df.style.set_table_styles([
        {'selector': 'th, td', 'props': [('border', '1px solid black'), ('padding', '8px')]}
    ])
    display(styled_df)

##### ZERO SHOT PROMPTING

In [46]:
user_input = input("describe your search below, give pass schema of tables that are involved in your search:\n")
response_openai = client_openai.responses.create(model='gpt-5.6-sol', input=user_input)
query = response_openai.output_text
print(f"""\n\nOPENAI: \n{query}""")



describe your search below, give pass schema of tables that are involved in your search:
 find all employees who are working in "Sales & Growth" department. you have 2 postgres tables whose schema are given below. give only sql query. table schema is. employees: "employee_id" "first_name" "last_name" "email" "department_id" "salary" "hire_date"   departments: "department_id" "department_name" "location"




OPENAI: 
SELECT e.*
FROM employees AS e
JOIN departments AS d
  ON e.department_id = d.department_id
WHERE d.department_name = 'Sales & Growth';


In [41]:
with conn.cursor() as cur:
    cur.execute(f"{query}")
    show_table(cur)
    

,employee_id,first_name,last_name,email,department_id,salary,hire_date
0,1,Alice,Smith,alice.smith@company.com,1,85000.00,2022-03-15


##### CONTEXT BASED PROMPTING

In [47]:
table_context = (f"""
TABLE 1: departments

Purpose: Stores information about company departments.

Columns:

department_id: SERIAL, PRIMARY KEY. Unique identifier for each department.
department_name: VARCHAR(100), NOT NULL. Name of the department.
location: VARCHAR(100), NOT NULL. Physical location of the department.

Sample data:

1, Sales & Growth, New York
2, Logistics & Ops, Chicago
3, Engineering, San Francisco

TABLE 2: employees

Purpose: Stores information about company employees.

Columns:

employee_id: SERIAL, PRIMARY KEY. Unique identifier for each employee.
first_name: VARCHAR(50), NOT NULL. Employee's first name.
last_name: VARCHAR(50), NOT NULL. Employee's last name.
email: VARCHAR(100), UNIQUE, NOT NULL. Employee's email address.
department_id: INT, FOREIGN KEY referencing departments.department_id.
salary: NUMERIC(10,2), NOT NULL. Employee salary.
hire_date: DATE, NOT NULL. Date the employee was hired.

Relationship:
employees.department_id references departments.department_id.

Sample data:

1, Alice, Smith, alice.smith@company.com, department 1, salary 85000.00, hired 2022-03-15
2, Bob, Jones, bob.jones@company.com, department 2, salary 72000.00, hired 2021-06-01
3, Charlie, Brown, charlie.brown@company.com, department 3, salary 115000.00, hired 2020-01-10

TABLE 3: customers

Purpose: Stores information about customers and their companies.

Columns:

customer_id: SERIAL, PRIMARY KEY. Unique identifier for each customer.
company_name: VARCHAR(150), NOT NULL. Name of the customer's company.
contact_name: VARCHAR(100), NOT NULL. Name of the primary customer contact.
country: VARCHAR(50), NOT NULL. Customer's country.
signup_date: DATE, NOT NULL. Date the customer signed up.

Sample data:

1, Alpha Corp, Jane Doe, USA, signup date 2023-01-10
2, Beta LLC, John Roe, Canada, signup date 2023-05-20
3, Gamma Inc, Acme User, UK, signup date 2024-02-14

TABLE 4: sales

Purpose: Stores individual sales transactions.

Columns:

sale_id: SERIAL, PRIMARY KEY. Unique identifier for each sale.
customer_id: INT, FOREIGN KEY referencing customers.customer_id.
employee_id: INT, FOREIGN KEY referencing employees.employee_id.
product_name: VARCHAR(100), NOT NULL. Name of the product sold.
quantity: INT, NOT NULL. Number of units sold.
unit_price: NUMERIC(10,2), NOT NULL. Price per unit.
sale_date: TIMESTAMP, DEFAULT CURRENT_TIMESTAMP. Date and time when the sale occurred.

Relationships:
sales.customer_id references customers.customer_id.
sales.employee_id references employees.employee_id.

Important calculation:
total_sale_amount = quantity * unit_price

Sample data:

Sale 1: customer 1, employee 1, Enterprise SaaS License, quantity 5, unit price 1200.00, sale date 2025-01-10 10:30:00
Sale 2: customer 2, employee 1, Cloud Hosting Add-on, quantity 10, unit price 150.00, sale date 2025-01-12 14:00:00
Sale 3: customer 3, employee 2, API Analytics Module, quantity 2, unit price 4500.00, sale date 2025-01-15 09:15:00
Sale 4: customer 1, employee 3, Security Audit Tool, quantity 1, unit price 3000.00, sale date 2025-01-18 16:45:00

TABLE 5: shipping

Purpose: Stores shipping and delivery information for sales.

Columns:

shipment_id: SERIAL, PRIMARY KEY. Unique identifier for each shipment.
sale_id: INT, FOREIGN KEY referencing sales.sale_id.
carrier: VARCHAR(50), NOT NULL. Shipping company/carrier.
shipping_cost: NUMERIC(10,2), NOT NULL. Cost of shipping.
delivery_status: VARCHAR(50), NOT NULL. Current delivery status.
shipped_date: DATE, nullable. Date the shipment was shipped.
delivered_date: DATE, nullable. Date the shipment was delivered.

Relationship:
shipping.sale_id references sales.sale_id.

Sample data:

Shipment 1: sale 1, FedEx, shipping cost 45.00, Delivered, shipped 2025-01-11, delivered 2025-01-13
Shipment 2: sale 2, UPS, shipping cost 25.50, Delivered, shipped 2025-01-13, delivered 2025-01-16
Shipment 3: sale 3, DHL, shipping cost 120.00, In Transit, shipped 2025-01-16, not delivered
Shipment 4: sale 4, FedEx, shipping cost 50.00, Pending, not shipped, not delivered

TABLE RELATIONSHIPS

employees.department_id → departments.department_id
sales.customer_id → customers.customer_id
sales.employee_id → employees.employee_id
shipping.sale_id → sales.sale_id

""")

In [48]:
def query_with_context(user_prompt, table_context):
    """Sends the user's prompt alongside the complete database schema context to OpenAI and returns the SQL query."""
    full_input = f"""
    Database Schema and Context:
    {table_context}
    
    User Request:
    {user_prompt}
    
    Provide only the SQL query as output.
    """
    
    response_openai = client_openai.responses.create(
        model='gpt-5.6-sol', 
        input=full_input
    )
    return response_openai.output_text

In [50]:
q1 = ("""Write a SQL query to display each sale with:
Customer company name
Employee full name
Product name
Quantity
Unit price
Total sale amount (quantity × unit_price)""")


q2 = ("""Write a SQL query to Find the total sales revenue generated by each employee.
Return:
Employee full name
Total revenue
Sort the results by total revenue from highest to lowest.""")



q3 = ("""Write a SQL query to Find all customers who have made purchases where the total value of an individual sale is greater than $5,000.
Return:
Customer company name
Product name
Quantity
Unit price
Total sale amount""")


q4 = ("""Write a SQL query to Find the monthly sales revenue for 2025.
Return:
Month
Total revenue
Sort chronologically by month.""")


q5 = ("""Write a SQL query to Find the department(s) whose employees generated more than $10,000 in total sales revenue.
Return:
Department name
Total revenue""")


In [60]:
# 1. Input your question number (e.g., 1, 2, 3...)
user_input = input("Enter query number (1-5): \n")

# Dynamically fetch the query text using globals()
selected_query = globals()[f"q{user_input}"]
print(f"User asked:\n{selected_query}")

# 2. Generate the SQL query using your context variable
query = query_with_context(selected_query, table_context)
print(f"\n\nOPENAI:\n{query}")

Enter query number (1-5): 
 1


User asked:
Write a SQL query to display each sale with:
Customer company name
Employee full name
Product name
Quantity
Unit price
Total sale amount (quantity × unit_price)


OPENAI:
SELECT
    c.company_name,
    CONCAT(e.first_name, ' ', e.last_name) AS employee_full_name,
    s.product_name,
    s.quantity,
    s.unit_price,
    s.quantity * s.unit_price AS total_sale_amount
FROM sales AS s
JOIN customers AS c
    ON c.customer_id = s.customer_id
JOIN employees AS e
    ON e.employee_id = s.employee_id;


In [62]:
# 3. Execute and show the styled table output
with conn.cursor() as cur:
    cur.execute(query)
    show_table(cur)

,company_name,employee_full_name,product_name,quantity,unit_price,total_sale_amount
0,Alpha Corp,Alice Smith,Enterprise SaaS License,5,1200.00,6000.00
1,Beta LLC,Alice Smith,Cloud Hosting Add-on,10,150.00,1500.00
2,Gamma Inc,Bob Jones,API Analytics Module,2,4500.00,9000.00
3,Alpha Corp,Charlie Brown,Security Audit Tool,1,3000.00,3000.00
4,Alpha Corp,Alice Smith,Enterprise SaaS License,5,1200.00,6000.00
5,Beta LLC,Alice Smith,Cloud Hosting Add-on,10,150.00,1500.00
6,Gamma Inc,Bob Jones,API Analytics Module,2,4500.00,9000.00
7,Alpha Corp,Charlie Brown,Security Audit Tool,1,3000.00,3000.00


In [69]:
def execute_query(query):
    # Pass the existing active connection directly to pandas
    df = pd.read_sql(query, conn)
    return df

In [70]:
# 1. Generate your query using AI and context as before
user_input = input("Enter query number (1-5): \n")
selected_query = globals()[f"q{user_input}"]

query = query_with_context(selected_query, table_context)
print(f"\n\nOPENAI:\n{query}")

# 2. Directly read the SQL query results into a DataFrame
execute_query(query)

Enter query number (1-5): 
 1




OPENAI:
SELECT
    c.company_name,
    CONCAT(e.first_name, ' ', e.last_name) AS employee_full_name,
    s.product_name,
    s.quantity,
    s.unit_price,
    s.quantity * s.unit_price AS total_sale_amount
FROM sales AS s
JOIN customers AS c
    ON c.customer_id = s.customer_id
JOIN employees AS e
    ON e.employee_id = s.employee_id;


/tmp/ipykernel_4221/1705314782.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,company_name,employee_full_name,product_name,quantity,unit_price,total_sale_amount
0,Alpha Corp,Alice Smith,Enterprise SaaS License,5,1200.0,6000.0
1,Beta LLC,Alice Smith,Cloud Hosting Add-on,10,150.0,1500.0
2,Gamma Inc,Bob Jones,API Analytics Module,2,4500.0,9000.0
3,Alpha Corp,Charlie Brown,Security Audit Tool,1,3000.0,3000.0
4,Alpha Corp,Alice Smith,Enterprise SaaS License,5,1200.0,6000.0
5,Beta LLC,Alice Smith,Cloud Hosting Add-on,10,150.0,1500.0
6,Gamma Inc,Bob Jones,API Analytics Module,2,4500.0,9000.0
7,Alpha Corp,Charlie Brown,Security Audit Tool,1,3000.0,3000.0
